# QBioCode-ADMET: Analysis and Decision Rule Derivation

**Phase 5 — Paper Figures and QSage Interpretation**

This notebook:
1. Loads compiled QProfiler results and test-set metrics
2. Generates the paper's main figures (heatmap, complexity scatter, SHAP)
3. Derives the empirical QML decision rule from QSage SHAP values
4. Produces the TDC leaderboard comparison table

**Prerequisites:** Run scripts 01–04 first.

In [ ]:
import os, sys
sys.path.insert(0, os.path.join(os.getcwd(), '..', '..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import dill

# Paths — adjust if needed
RESULTS_DIR  = '../../results/admet_benchmark'
SAGE_DIR     = f'{RESULTS_DIR}/sage'
TEST_DIR     = f'{RESULTS_DIR}/test_results'
FIGURES_DIR  = f'{RESULTS_DIR}/figures'
os.makedirs(FIGURES_DIR, exist_ok=True)
print('Paths configured.')

## 1. Load Results

In [ ]:
# Compiled QProfiler model results
model_df = pd.read_csv(f'{SAGE_DIR}/compiled_ModelResults.csv')
# Test-set metrics (from 04_test_inference.py)
test_df  = pd.read_csv(f'{TEST_DIR}/test_results.csv')
# Trained QSage
with open(f'{SAGE_DIR}/trained_sage.pkl', 'rb') as f:
    sage = dill.load(f)

print(f'Model results : {model_df.shape}')
print(f'Test results  : {test_df.shape}')
print(f'Models in sage: {sage._available_models}')

## 2. Figure 1 — AUROC Heatmap (Endpoints × Models)

In [ ]:
pivot = test_df.groupby(['endpoint', 'model'])['auroc'].mean().unstack('model')

fig, ax = plt.subplots(figsize=(14, 9))
sns.heatmap(
    pivot, annot=True, fmt='.3f', cmap='RdYlGn',
    vmin=0.5, vmax=1.0, linewidths=0.5, ax=ax
)
ax.set_title('AUROC: Endpoints × Models (mean over featurizers × splits)', fontsize=13)
ax.set_xlabel('Model')
ax.set_ylabel('ADMET Endpoint')
plt.tight_layout()
fig.savefig(f'{FIGURES_DIR}/fig1_auroc_heatmap.pdf', bbox_inches='tight')
plt.show()

## 3. Figure 2 — Complexity vs. ΔF1 (QML − Classical)

In [ ]:
# Load complexity metrics
eval_df = pd.read_csv(f'{SAGE_DIR}/compiled_RawDataEvaluation.csv')

# QML models
qml_models = ['qsvc', 'vqc', 'qnn', 'pqk']
classical_models = ['svc', 'rf', 'mlp', 'xgb', 'lr']

# Compute best QML F1 and best classical F1 per endpoint
qml_f1 = (
    test_df[test_df['model'].isin(qml_models)]
    .groupby('endpoint')['f1'].max().rename('qml_f1')
)
cls_f1 = (
    test_df[test_df['model'].isin(classical_models)]
    .groupby('endpoint')['f1'].max().rename('cls_f1')
)
delta_df = pd.concat([qml_f1, cls_f1], axis=1).dropna()
delta_df['delta_f1'] = delta_df['qml_f1'] - delta_df['cls_f1']

# Merge with complexity
complexity_mean = eval_df.groupby('Dataset')[['Intrinsic_Dimension', '# Samples']].mean()
delta_df = delta_df.join(complexity_mean, how='left')

fig, ax = plt.subplots(figsize=(8, 6))
sc = ax.scatter(
    delta_df['Intrinsic_Dimension'], delta_df['delta_f1'],
    c=delta_df['# Samples'], cmap='viridis_r', s=80, edgecolors='k', linewidth=0.5
)
ax.axhline(0, color='gray', linestyle='--', linewidth=1)
ax.set_xlabel('Intrinsic Dimension')
ax.set_ylabel('ΔF1 (best QML − best Classical)')
ax.set_title('QML advantage vs. Dataset Complexity')
plt.colorbar(sc, ax=ax, label='# Training Samples')
# Annotate endpoints where QML wins
for ep, row in delta_df[delta_df['delta_f1'] > 0].iterrows():
    ax.annotate(ep, (row['Intrinsic_Dimension'], row['delta_f1']),
                fontsize=7, xytext=(3, 3), textcoords='offset points')
plt.tight_layout()
fig.savefig(f'{FIGURES_DIR}/fig2_complexity_vs_delta_f1.pdf', bbox_inches='tight')
plt.show()

## 4. Figure 3 — QSage SHAP Feature Importance

In [ ]:
import shap

# Pick best-performing QML model × f1_score metric for SHAP
target_model = 'qsvc'
target_metric = 'f1_score'

result = sage._results_subsages[target_metric][target_model]
fitted_xgb = result['fit_model']

model_indices = sage._input_data_metadata[
    sage._input_data_metadata['model'] == target_model
].index
X = sage._input_data_features_only.loc[model_indices]
X = X.replace([float('inf'), float('-inf')], float('nan')).fillna(0)

explainer = shap.TreeExplainer(fitted_xgb)
shap_values = explainer.shap_values(X)

plt.figure(figsize=(9, 6))
shap.summary_plot(shap_values, X, show=False, max_display=15)
plt.title(f'QSage SHAP — {target_model} / {target_metric}')
plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}/fig3_shap_qsage.pdf', bbox_inches='tight')
plt.show()

## 5. Decision Rule Derivation

In [ ]:
from sklearn.tree import DecisionTreeClassifier, export_text

# Binary label: QML wins (delta_f1 > 0)
delta_df['qml_wins'] = (delta_df['delta_f1'] > 0).astype(int)
features = ['Intrinsic_Dimension', '# Samples']
X_rule = delta_df[features].dropna()
y_rule = delta_df.loc[X_rule.index, 'qml_wins']

# Fit a depth-2 decision stump for interpretability
stump = DecisionTreeClassifier(max_depth=2, random_state=42)
stump.fit(X_rule, y_rule)

print('Decision Rule:')
print(export_text(stump, feature_names=features))
print(f'\nRule accuracy: {stump.score(X_rule, y_rule):.3f}')

## 6. TDC Leaderboard Comparison

In [ ]:
# Best per endpoint × model
best_test = (
    test_df.groupby(['endpoint', 'model'])['auroc']
    .mean()
    .reset_index()
    .sort_values(['endpoint', 'auroc'], ascending=[True, False])
)
best_per_endpoint = best_test.groupby('endpoint').first().reset_index()
best_per_endpoint.columns = ['endpoint', 'best_model', 'auroc']
print(best_per_endpoint.to_string(index=False))
best_per_endpoint.to_csv(f'{TEST_DIR}/tdc_leaderboard_comparison.csv', index=False)
print(f'\nSaved to {TEST_DIR}/tdc_leaderboard_comparison.csv')